```sh
pip install sagemaker-core ipywidgets
```

### Notes

- Sagemaker container available support until python 3.12
- Referencia: https://github.com/aws/deep-learning-containers/blob/master/available_images.md

- Instance types to test: ml.g4dn.xlarge, ml.g6.xlarge, ml.trn1.2xlarge 
- https://github.com/aws/amazon-sagemaker-examples/blob/default/%20%20%20%20%20%20%20%20%20sagemaker-core/introductory-blog-sagemaker-core/introductory-blog-sagemaker-core.ipynb

In [ ]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session, s3_path_join

# Set up region, role and bucket parameters used throughout the notebook.
sagemaker_session = Session(
    default_bucket_prefix='congestion-bogota-training'
)
# region = sagemaker_session.boto_region_name
# role = get_execution_role()
region = 'us-east-1'
role = 'arn:aws:iam::195419001736:role/SagemakerExecutionRole'
bucket_name = "sagemaker-congestion-bogota"

print(f"AWS region: {region}")
print(f"Execution role: {role}")
print(f"Default S3 bucket: {bucket_name}")

In [ ]:
import os
import tempfile
import tarfile
import boto3

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'


def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    '''Compress only the training code and upload to S3.'''

    include_files = [
        'requirements-sagemaker.txt',
    ]

    with tempfile.TemporaryDirectory() as tmp_dir_name:
        tar_file_name = os.path.join(tmp_dir_name, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_file_name, mode='w:gz', dereference=True) as tar:
            src_root = os.path.join(source_path, 'src')
            for root, dirs, files in os.walk(src_root):
                dirs[:] = [d for d in dirs if d != '__pycache__']
                for fname in files:
                    if not fname.endswith('.py'):
                        continue
                    abs_path = os.path.join(root, fname)
                    rel_path = os.path.relpath(abs_path, source_path)
                    tar.add(abs_path, arcname=rel_path)

            for rel_path in include_files:
                abs_path = os.path.join(source_path, rel_path)
                if os.path.exists(abs_path):
                    tar.add(abs_path, arcname=rel_path)

        s3_key = '/'.join([s3_key_prefix, TAR_SOURCE_FILENAME])
        boto3.client('s3').upload_file(tar_file_name, bucket, s3_key)
        return s3_path_join('s3://', bucket, s3_key)

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/code')


In [ ]:
image = "195419001736.dkr.ecr.us-east-1.amazonaws.com/congestion-bogota-sagemaker:temporal-features-20260331-194332"

model_configs = [
    {"model_type": "GConvGRU", "instance_type": "ml.g5.4xlarge"},
    {"model_type": "GConvLSTM", "instance_type": "ml.g4dn.8xlarge"},
    {"model_type": "TGCN", "instance_type": "ml.g5.8xlarge"},
]

instance_count = 1
volume_size_in_gb = 100
max_runtime_in_seconds = 36 * 60 * 60
project_s3_path = s3_path_join("s3://", bucket_name, "congestion-bogota")

seed_time_artifact_path = (
    "s3://sagemaker-congestion-bogota/congestion-bogota/seed-time-artifacts/"
    "final_all/seed_time_bucket_rows_merged_all.parquet"
)

shape_loss_config = {
    "loss-shape-lambda": "0.05",
    "loss-shape-min-std": "1e-3",
    "loss-shape-eps": "1e-6",
    "loss-shape-weight-std": "0.75",
    "loss-shape-weight-range": "0.25",
    "loss-shape-weight-max": "1.0",
}

operational_loss_config = {
    "loss-horizon-short-steps": "16",
    "loss-horizon-mid-steps": "32",
    "loss-horizon-short-weight": "1.5",
    "loss-horizon-mid-weight": "1.0",
    "loss-horizon-long-weight": "0.6",
    "loss-zero-overprediction-lambda": "0.4",
    "loss-zero-overprediction-threshold": "0.5",
    "loss-high-underprediction-lambda": "0.6",
    "loss-high-underprediction-threshold": "3",
}

base_hyper_parameters = {
    "epochs": "2",
    "batch-size": "16",
    "learning-rate": "0.001",
    "input-seq-len": "150",
    "output-seq-len": "50",
    "node-batch-size": "500",
    "train-ratio": "0.8",
    "include-weather": "True",
    "include-velocity-target": "True",
    "seeded-batch-sampler": "True",
    "seed-time-artifact-path": seed_time_artifact_path,
    "seed-time-bucket-target-ratios": "0.45,0.30,0.25",
    "seed-time-epoch-cycles": "1",
    "seed-time-shuffle-seeds": "True",
    "num-workers": "4",
    "drop-last": "False",
    "shuffle-time": "True",
    "seeded-batch-seed": "42",
    "pin-memory": "True",
    "prefetch-factor": "2",
    "persistent-workers": "True",
    "cache-local": "True",
    "cache-dir": "/opt/ml/input/data/cache",
    "use-minimal-datasets": "True",
    "sagemaker_submit_directory": source_code_uri,
    "sagemaker_program": "src/training_entrypoint.py",
    "loss-continuous-type": "huber",
    "loss-huber-delta": "1.0",
    "loss-class-ce-lambda": "0.3",
    "loss-class-distance-lambda": "0.5",
    "loss-large-error-threshold": "2",
    "loss-large-error-lambda": "0.75",
    "loss-high-level-threshold": "2",
    "loss-high-level-lambda": "0.5",
    "loss-piecewise-threshold": "4",
    "loss-piecewise-mae-lambda": "0.5",
    "loss-directional-lambda": "0.3",
    "timing-steps": "20",
}
base_hyper_parameters.update(shape_loss_config)
base_hyper_parameters.update(operational_loss_config)



In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=region)

NUMERIC_REGEX = r"([-+0-9eE\.]+)"

metric_names = [
    "train_loss",
    "loss_level",
    "loss_velocity",
    "train_probe_mae_level",
    "train_probe_accuracy_exact",
    "train_probe_accuracy_pm1",
    "train_probe_class_distance_mean",
    "train_probe_class_distance_ge2_rate",
]

for level in range(6):
    metric_names.extend([
        f"train_probe_level_{level}_count",
        f"train_probe_level_{level}_mae",
        f"train_probe_level_{level}_accuracy_exact",
    ])

assert len(metric_names) <= 40, f"Too many metric definitions: {len(metric_names)}"

metric_definitions_payload = [
    {"Name": name, "Regex": rf"{name}={NUMERIC_REGEX}"}
    for name in metric_names
]

launched_jobs = []

for cfg in model_configs:
    model_type = cfg["model_type"]
    instance_type = cfg["instance_type"]
    job_name = f"congestion-bogota-{model_type.lower()}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
    s3_output_path = s3_path_join(project_s3_path, "training-runs", model_type.lower())
    hyper_parameters = dict(base_hyper_parameters)
    hyper_parameters["model-type"] = model_type

    training_job_request = {
        "TrainingJobName": job_name,
        "RoleArn": role,
        "AlgorithmSpecification": {
            "TrainingImage": image,
            "TrainingInputMode": "File",
            "MetricDefinitions": metric_definitions_payload,
            "EnableSageMakerMetricsTimeSeries": True,
        },
        "HyperParameters": hyper_parameters,
        "InputDataConfig": [
            {
                "ChannelName": "train",
                "DataSource": {
                    "FileSystemDataSource": {
                        "FileSystemId": efs_id,
                        "FileSystemType": "EFS",
                        "FileSystemAccessMode": "ro",
                        "DirectoryPath": efs_path,
                    }
                },
            },
            {
                "ChannelName": "validation",
                "DataSource": {
                    "FileSystemDataSource": {
                        "FileSystemId": efs_id,
                        "FileSystemType": "EFS",
                        "FileSystemAccessMode": "ro",
                        "DirectoryPath": efs_path,
                    }
                },
            },
        ],
        "OutputDataConfig": {
            "S3OutputPath": s3_output_path,
        },
        "ResourceConfig": {
            "InstanceType": instance_type,
            "InstanceCount": instance_count,
            "VolumeSizeInGB": volume_size_in_gb,
        },
        "StoppingCondition": {
            "MaxRuntimeInSeconds": max_runtime_in_seconds,
        },
        "VpcConfig": {
            "Subnets": vpc_config["subnets"],
            "SecurityGroupIds": vpc_config["security_group_ids"],
        },
    }

    response = sm.create_training_job(**training_job_request)
    launched_jobs.append(
        {
            "model_type": model_type,
            "instance_type": instance_type,
            "job_name": job_name,
            "training_job_arn": response["TrainingJobArn"],
        }
    )

launched_jobs



In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=region)
desc = sm.describe_training_job(TrainingJobName=job_name)
print(desc["TrainingJobStatus"])
print(desc["SecondaryStatus"])
print(desc.get("FailureReason"))
